# Seasonal and weather effects on bike-sharing demand
### A two-way ANOVA project

Tools: Python, Pandas, Matplotlib, Seaborn, SciPy, Statsmodels
Statistical method: two-way factorial ANOVA
Dataset: UCI Bike sharing dataset (2011-2012)

Business objective: investigate whether average daily bike_sharing demand differs significantly across seasons and weather conditions.

## Reasearch questions and hypotheses
### Research questions
1. Does average daily rental demand differ significantly across seasons?
2. Does average daily rental demand differ between clear anf misty/cloudy weather conditions?
3. Does the relationship between weather conditions and rental demand very across seasons?

### Statistical hypotheses
H1: main effect of season
- H0: the adjusted mean rental demand is equal across seasons
- H1: at least one seasonal adjusted mean differs

H2: main effect of weather
- H0: the adjusted mean demand is equal between wether groups
- H1: the adjusted mean demand differs between weather groups

H3: season x weather interaction
- H0: the difference between weather groups is the same across all seasons
- H1: the difference between weather groups varies by season

Significance level (alpha) = 0.50

## Dataset description

### Data source
Dataset: Bike sharing dataset

Source: UCI Machine learning repository

Period : January 2011 - December 2012

URL: https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset

The dataset contain historical bike rental counts together with weather and calendar information.

### Variables of interest
| Variable | Description | Analytical Role |
|---|---|---|
| `cnt` | Total daily bike rentals | Dependent variable |
| `season` | Season category | Independent variable |
| `weathersit` | Weather category | Independent variable |
| `dteday` | Observation date | Time identifier |
| `yr` | Year of observation | Potential control |
| `workingday` | Workday indicator | Potential control |

### Analytical scope
The primary analysis will compare four seasons and two weather categories: clear and misty/cloudy conditions (to avoid relying on very sparcely populated groups)

### 1. Data loading and initial inspection

In [9]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

sns.set_theme(style="whitegrid")

cwd = Path.cwd()

if (cwd / "01_data" / "01_raw" / "day.csv").exists():
    project_root = cwd
elif (cwd.parent / "01_data" / "01_raw" / "day.csv").exists():
        project_root = cwd.parent
else:
    raise FileNotFoundError(
        "Could not find. "
        "Check your folder structure. "
    )
data_path = project_root / "01_data" / "01_raw" /"day.csv"

df = pd.read_csv(data_path, parse_dates=["dteday"])

print("Shape:", df.shape)
display(df.head())
df.info()

Shape: (731, 16)


,instant,dteday,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,6,0,2,0.344167,0.363625,0.805833,0.160446,331,654,985
1,2,2011-01-02,1,0,1,0,0,0,2,0.363478,0.353739,0.696087,0.248539,131,670,801
2,3,2011-01-03,1,0,1,0,1,1,1,0.196364,0.189405,0.437273,0.248309,120,1229,1349
3,4,2011-01-04,1,0,1,0,2,1,1,0.200000,0.212122,0.590435,0.160296,108,1454,1562
4,5,2011-01-05,1,0,1,0,3,1,1,0.226957,0.229270,0.436957,0.186900,82,1518,1600


<class 'pandas.DataFrame'>
RangeIndex: 731 entries, 0 to 730
Data columns (total 16 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   instant     731 non-null    int64         
 1   dteday      731 non-null    datetime64[us]
 2   season      731 non-null    int64         
 3   yr          731 non-null    int64         
 4   mnth        731 non-null    int64         
 5   holiday     731 non-null    int64         
 6   weekday     731 non-null    int64         
 7   workingday  731 non-null    int64         
 8   weathersit  731 non-null    int64         
 9   temp        731 non-null    float64       
 10  atemp       731 non-null    float64       
 11  hum         731 non-null    float64       
 12  windspeed   731 non-null    float64       
 13  casual      731 non-null    int64         
 14  registered  731 non-null    int64         
 15  cnt         731 non-null    int64         
dtypes: datetime64[us](1), float64(4), int

### 2. Data quality assessment

In [10]:
#checking the data
print("Rows and columns:", df.shape)

print("\nMissing values:")
print(df.isna().sum())

print("\nDuplicate dates:", df["dteday"].duplicated().sum())

print("\nNegative rental counts:", (df["cnt"] < 0).sum())

print(
    "\nInconsistent totals:",
    (df["cnt"] != df["casual"] + df["registered"]).sum()
)

print("\nSeasons:")
print(df["season"].value_counts().sort_index())

Rows and columns: (731, 16)

Missing values:
instant       0
dteday        0
season        0
yr            0
mnth          0
holiday       0
weekday       0
workingday    0
weathersit    0
temp          0
atemp         0
hum           0
windspeed     0
casual        0
registered    0
cnt           0
dtype: int64

Duplicate dates: 0

Negative rental counts: 0

Inconsistent totals: 0

Seasons:
season
1    181
2    184
3    188
4    178
Name: count, dtype: int64


### 3. Data preparation


**Factor A: Season**
- Winter
- Spring
- Summer
- Fall

**Factor B: Weather**
- Clear
- Misty / Cloudy

**Dependent Variable:**
- Total daily rental count (`cnt`)

In [11]:
#creating readable categories
season_map = {1: "Winter", 2: "Spring", 3: "Summer", 4: "Fall"}
weather_map = {1: "Clear", 2: "Misty / Cloudy", 3: "Light precipitation", 4:"Severe weather"}

df["season_label"] = df["season"].map(season_map)
df["weather_label"] = df["weathersit"].map(weather_map)

season_order = ["Winter", "Spring", "Summer", "Fall"]
weather_order = ["Clear", "Misty / Cloudy"]

assert df["season_label"]. notna().all()
assert df["weather_label"].notna().all()

display(
    df[["dteday", "season_label", "weather_label", "cnt"]].head()
)

,dteday,season_label,weather_label,cnt
0,2011-01-01,Winter,Misty / Cloudy,985
1,2011-01-02,Winter,Misty / Cloudy,801
2,2011-01-03,Winter,Clear,1349
3,2011-01-04,Winter,Clear,1562
4,2011-01-05,Winter,Clear,1600


In [12]:
#checking how many observations are for each combination of factors
cell_counts = pd.crosstab(
    df["season_label"],
    df["weather_label"]
)

display(cell_counts)

weather_label,Clear,Light precipitation,Misty / Cloudy
season_label,,,
Fall,103,10,65
Spring,113,3,68
Summer,136,4,48
Winter,111,4,66


In [13]:
# primary anova
anova_df = df[df["weathersit"].isin([1,2])].copy()

design_counts = pd.crosstab(
    anova_df["season_label"],
    anova_df["weather_label"]
).reindex(
    index=season_order,
    columns=weather_order,
    fill_value=0
)

display(design_counts)

assert (design_counts >= 10).all().all(), (
    "Some groups contain fewer than 10 days. "
    "Review the factorial design."
)

print("Observations used: ", len(anova_df))
print("Observations excluded: ", len(df) - len(anova_df))

weather_label,Clear,Misty / Cloudy
season_label,,
Winter,111,66
Spring,113,68
Summer,136,48
Fall,103,65


Observations used:  710
Observations excluded:  21
